In [1]:
import os
import numpy as np
from PIL import Image
from skimage import measure
from scipy.spatial.distance import directed_hausdorff
from tqdm import tqdm
import matplotlib.pyplot as plt

In [13]:
#Google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def calculate_challenge_metrics(manual_mask, auto_mask):

    manual_mask = np.array(manual_mask) > 0
    auto_mask   = np.array(auto_mask) > 0

    TP = np.sum(manual_mask & auto_mask)
    FP = np.sum((~manual_mask) & auto_mask)
    FN = np.sum(manual_mask & (~auto_mask))
    TN = np.sum((~manual_mask) & (~auto_mask))

    # ===== IoU =====
    union = TP + FP + FN
    IoU = TP / union if union > 0 else np.nan

    # ===== Dice =====
    denom_dice = 2*TP + FP + FN
    Dice = 2*TP / denom_dice if denom_dice > 0 else np.nan

    # ===== Precision =====
    Precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan

    # ===== Recall =====
    Recall = TP / (TP + FN) if (TP + FN) > 0 else np.nan

    # ===== Accuracy =====
    Accuracy = (TP + TN) / (TP + TN + FP + FN)

    # ===== Count Error =====
    labels_man = measure.label(manual_mask, connectivity=2)
    labels_auto = measure.label(auto_mask, connectivity=2)
    count_error = abs(labels_man.max() - labels_auto.max())

    # ===== Steatosis Error =====
    area = manual_mask.size
    sp_manual = manual_mask.sum() / area * 100
    sp_auto   = auto_mask.sum() / area * 100
    steatosis_error = abs(sp_manual - sp_auto)

    # ===== Hausdorff =====
    coords_man  = np.column_stack(np.nonzero(manual_mask))
    coords_auto = np.column_stack(np.nonzero(auto_mask))

    if coords_man.size > 0 and coords_auto.size > 0:
        hd1 = directed_hausdorff(coords_man, coords_auto)[0]
        hd2 = directed_hausdorff(coords_auto, coords_man)[0]
        HD = max(hd1, hd2)
    else:
        HD = np.nan

    return IoU, Dice, Precision, Recall, Accuracy, HD, count_error, steatosis_error


In [16]:
# ================================================================
# ABLATION STUDY – POST PROCESSING (SOLO ISOLATA, SENZA COMBINAZIONI)
# (METRICHE IDENTICHE ALLA TUA PIPELINE + NaN/casi + micro-avg)
# Ogni variante parte SEMPRE da auto_raw
# Varianti: baseline, only_remove, only_fill, only_open, only_close
# ================================================================

import os
import numpy as np
from tqdm import tqdm
from PIL import Image

from scipy import ndimage
from skimage.morphology import (
    remove_small_objects,
    binary_closing,
    binary_opening,
    disk
)

# ------------------------------------------------
# PATH
# ------------------------------------------------
current_dir = '/content/drive/MyDrive/MACHENKO_ONLY/dataset_diviso_PROCESSED'
GT_MASK_DIR   = os.path.join(current_dir, 'TS', 'manual')
PRED_MASK_DIR = os.path.join(current_dir, 'TEST_PRED')

# ------------------------------------------------
# PARAMETRI POST-PROCESSING
# ------------------------------------------------
MIN_AREA   = 20
OPENING_R  = 1
CLOSING_R  = 1

# ------------------------------------------------
# POST-PROCESSING: ISOLATO (OGNI VARIANTE PARTE DA auto_raw)
# ------------------------------------------------
def postprocess_isolated(pred, variant):
    """
    variant:
    baseline, only_remove, only_fill, only_open, only_close
    """
    out = pred.copy().astype(bool)

    if variant == "baseline":
        pass

    elif variant == "only_remove":
        out = remove_small_objects(out, min_size=MIN_AREA)

    elif variant == "only_fill":
        out = ndimage.binary_fill_holes(out)

    elif variant == "only_open":
        out = binary_opening(out, disk(OPENING_R))

    elif variant == "only_close":
        out = binary_closing(out, disk(CLOSING_R))

    else:
        raise ValueError(f"Unknown variant: {variant}")

    return out.astype(np.uint8)

# ------------------------------------------------
# FILE LIST (BASATA SULLE GT)
# ------------------------------------------------
filenames = sorted([f for f in os.listdir(GT_MASK_DIR) if f.endswith('.png')])
print(f"Inizio ablation ISOLATA su {len(filenames)} immagini")

metric_names = ["IoU","Dice","Precision","Recall","Accuracy","HD","CountError","SteatosisError"]

# ------------------------------------------------
# STRUTTURE RISULTATI: ISOLATA (NO OC/CO)
# ------------------------------------------------
iso_variants = ["baseline","only_remove","only_fill","only_open","only_close"]
iso_labels = {
    "baseline"   : "ISO | Baseline (no post)",
    "only_remove": "ISO | Only remove small objects",
    "only_fill"  : "ISO | Only hole filling",
    "only_open"  : "ISO | Only opening (Erosion -> Dilation)",
    "only_close" : "ISO | Only closing (Dilation -> Erosion)",
}

iso_metrics = {v: {k: [] for k in metric_names} for v in iso_variants}
iso_case_counts = {v: {"empty_empty": 0, "gt_empty_pred_pos": 0, "gt_pos_pred_empty": 0, "pos_pos": 0} for v in iso_variants}
iso_nan_counts  = {v: {k: 0 for k in metric_names} for v in iso_variants}
iso_micro_conf  = {v: {"TP": 0, "FP": 0, "FN": 0, "TN": 0} for v in iso_variants}

# ------------------------------------------------
# HELPER: aggiorna contatori casi + micro + NaN
# ------------------------------------------------
def update_stats(key_id, manual_mask, pred_mask, metrics_store, case_store, nan_store, micro_store, metric_values):
    man_bin = (manual_mask > 0)
    aut_bin = (pred_mask > 0)

    man_any = man_bin.any()
    aut_any = aut_bin.any()

    if (not man_any) and (not aut_any):
        case_store[key_id]["empty_empty"] += 1
    elif (not man_any) and aut_any:
        case_store[key_id]["gt_empty_pred_pos"] += 1
    elif man_any and (not aut_any):
        case_store[key_id]["gt_pos_pred_empty"] += 1
    else:
        case_store[key_id]["pos_pos"] += 1

    TP = np.sum(man_bin & aut_bin)
    FP = np.sum((~man_bin) & aut_bin)
    FN = np.sum(man_bin & (~aut_bin))
    TN = np.sum((~man_bin) & (~aut_bin))

    micro_store[key_id]["TP"] += TP
    micro_store[key_id]["FP"] += FP
    micro_store[key_id]["FN"] += FN
    micro_store[key_id]["TN"] += TN

    for k, v in zip(metric_names, metric_values):
        metrics_store[key_id][k].append(v)
        if np.isnan(v):
            nan_store[key_id][k] += 1

# ------------------------------------------------
# LOOP PRINCIPALE
# ------------------------------------------------
for f_name in tqdm(filenames, desc="Ablation (iso only)"):

    path_man = os.path.join(GT_MASK_DIR, f_name)

    base = f_name.replace("_preproc.png", "")
    auto_name = base + "_preproc.png"
    path_aut = os.path.join(PRED_MASK_DIR, auto_name)

    if not os.path.exists(path_aut):
        continue

    manual_mask = np.array(Image.open(path_man).convert('L'))
    auto_raw    = np.array(Image.open(path_aut).convert('L'))
    auto_raw    = (auto_raw > 0).astype(np.uint8)

    for var in iso_variants:
        auto_pp = postprocess_isolated(auto_raw, var)

        vals = calculate_challenge_metrics(manual_mask, auto_pp)  # funzione già definita nel tuo notebook
        update_stats(var, manual_mask, auto_pp,
                     iso_metrics, iso_case_counts, iso_nan_counts, iso_micro_conf, vals)

# ------------------------------------------------
# STAMPA RISULTATI (Mean±Std + NaN + casi + micro-avg)
# ------------------------------------------------
def print_block(title, keys, labels, metrics_store, case_store, nan_store, micro_store):
    print("\n" + "="*70)
    print(title)
    print("="*70)

    for key_id in keys:
        print(f"\n{labels[key_id]}")

        N = len(metrics_store[key_id]["Accuracy"])

        for m in metric_names:
            values = metrics_store[key_id][m]
            mean_val = np.nanmean(values)
            std_val  = np.nanstd(values)
            print(f"{m:15}: {mean_val:.4f} ± {std_val:.4f}   | NaN: {nan_store[key_id][m]}/{N}")

        print("  -- Casi (GT vs Pred) --")
        for k, v in case_store[key_id].items():
            pct = (v / N * 100) if N > 0 else 0
            print(f"  {k:18}: {v} ({pct:.1f}%)")

        TPg = micro_store[key_id]["TP"]
        FPg = micro_store[key_id]["FP"]
        FNg = micro_store[key_id]["FN"]
        TNg = micro_store[key_id]["TN"]

        union_g = TPg + FPg + FNg
        iou_micro  = TPg / union_g if union_g > 0 else np.nan
        dice_micro = 2*TPg / (2*TPg + FPg + FNg) if (2*TPg + FPg + FNg) > 0 else np.nan
        prec_micro = TPg / (TPg + FPg) if (TPg + FPg) > 0 else np.nan
        rec_micro  = TPg / (TPg + FNg) if (TPg + FNg) > 0 else np.nan
        acc_micro  = (TPg + TNg) / (TPg + TNg + FPg + FNg) if (TPg + TNg + FPg + FNg) > 0 else np.nan

        print("  -- Micro-averaged --")
        print(f"  IoU_micro       : {iou_micro:.4f}")
        print(f"  Dice_micro      : {dice_micro:.4f}")
        print(f"  Precision_micro : {prec_micro:.4f}")
        print(f"  Recall_micro    : {rec_micro:.4f}")
        print(f"  Accuracy_micro  : {acc_micro:.4f}")

# ------------------------------------------------
# OUTPUT
# ------------------------------------------------
print_block(
    title="ABLATION ISOLATA (solo un blocco, NO OC/CO) – METRICHE CHALLENGE (+ NaN/casi/micro)",
    keys=iso_variants,
    labels=iso_labels,
    metrics_store=iso_metrics,
    case_store=iso_case_counts,
    nan_store=iso_nan_counts,
    micro_store=iso_micro_conf
)


Inizio ablation ISOLATA su 56 immagini


Ablation (iso only): 100%|██████████| 56/56 [00:04<00:00, 11.27it/s]


ABLATION ISOLATA (solo un blocco, NO OC/CO) – METRICHE CHALLENGE (+ NaN/casi/micro)

ISO | Baseline (no post)
IoU            : 0.4878 ± 0.2319   | NaN: 19/56
Dice           : 0.6174 ± 0.2489   | NaN: 19/56
Precision      : 0.6593 ± 0.2672   | NaN: 19/56
Recall         : 0.6652 ± 0.2165   | NaN: 21/56
Accuracy       : 0.9917 ± 0.0123   | NaN: 0/56
HD             : 101.7944 ± 84.8603   | NaN: 21/56
CountError     : 4.3750 ± 5.5245   | NaN: 0/56
SteatosisError : 0.2222 ± 0.3260   | NaN: 0/56
  -- Casi (GT vs Pred) --
  empty_empty       : 19 (33.9%)
  gt_empty_pred_pos : 2 (3.6%)
  gt_pos_pred_empty : 0 (0.0%)
  pos_pos           : 35 (62.5%)
  -- Micro-averaged --
  IoU_micro       : 0.6735
  Dice_micro      : 0.8049
  Precision_micro : 0.8297
  Recall_micro    : 0.7816
  Accuracy_micro  : 0.9917

ISO | Only remove small objects
IoU            : 0.4990 ± 0.2223   | NaN: 20/56
Dice           : 0.6316 ± 0.2336   | NaN: 20/56
Precision      : 0.6796 ± 0.2492   | NaN: 20/56
Recall         :